# 09 — Distillation on Colab (soft KD, 6 labels)

**Teacher:** `unitary/toxic-bert` (frozen)  
**Student:** `distilbert-base-uncased` → 6-label multi-label head  
**Loss:** α · KD(soft teacher probs) + (1−α) · BCE(hard Jigsaw labels)

## Colab steps
1. Runtime → Change runtime type → **GPU**
2. Upload this repo (or clone from GitHub) so `src/` and `data/raw/jigsaw/` are available
3. Run all cells
4. Download `models/distilbert-student/` zip back to your laptop
5. Locally: `python scripts/run_model_comparison.py --models toxic-bert tfidf-svm distilbert-student --eval-n 500`

Local CPU smoke (not for submission quality):
```bash
python scripts/train_distill.py --max-train-samples 2000 --max-val-samples 500 --epochs 1
```

In [ ]:
# Colab: uncomment if needed
# !pip install -q transformers datasets scikit-learn pandas pyarrow accelerate

import os, sys
from pathlib import Path

# Point at repo root (adjust if you uploaded elsewhere)
ROOT = Path.cwd()
if (ROOT / "src").exists():
    pass
elif (ROOT / "DeepProjectHate" / "src").exists():
    ROOT = ROOT / "DeepProjectHate"
elif (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)
print("ROOT", ROOT.resolve())

In [ ]:
from device import get_device
print("device:", get_device())

# Ensure data + splits exist
from pathlib import Path
from paths import TRAIN_CSV, SPLIT_INDICES_PATH
if not TRAIN_CSV.exists():
    print("Downloading Jigsaw...")
    %run scripts/download_jigsaw.py
if not SPLIT_INDICES_PATH.exists():
    %run scripts/make_splits.py
if not (Path("models") / "toxic-bert").exists():
    print("Downloading toxic-bert teacher...")
    %run scripts/download_toxic_bert.py

In [ ]:
# Full Colab run — drop max_* caps for submission-quality student
MAX_TRAIN = None   # e.g. 20000 for a faster Colab pass; None = full train split
MAX_VAL = None
EPOCHS = 2
BATCH = 32

cmd = f"python scripts/train_distill.py --epochs {EPOCHS} --batch-size {BATCH}"
if MAX_TRAIN:
    cmd += f" --max-train-samples {MAX_TRAIN}"
if MAX_VAL:
    cmd += f" --max-val-samples {MAX_VAL}"
print(cmd)
!{cmd}

In [ ]:
import json
from paths import LOCAL_DISTILBERT_STUDENT_DIR, DISTILL_RESULTS_PATH
print("Student dir:", LOCAL_DISTILBERT_STUDENT_DIR, "exists=", LOCAL_DISTILBERT_STUDENT_DIR.exists())
if DISTILL_RESULTS_PATH.exists():
    print(json.dumps(json.loads(DISTILL_RESULTS_PATH.read_text()), indent=2)[:1500])

# Zip for download from Colab
!zip -r distilbert-student.zip models/distilbert-student
print("Download distilbert-student.zip from the file browser, extract to models/distilbert-student locally.")